In [ ]:
# Parameters
input_image = None  # papermill will inject the path


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from radiomics import featureextractor
import SimpleITK as sitk
from PIL import Image
print('radiomics notebook')


In [ ]:
in_img = Path(input_image)
mask_candidates = list(Path('notebooks/output_oct').glob(f'mask_{in_img.stem}*.png'))
if not mask_candidates:
    raise FileNotFoundError('Mask not found for ' + str(in_img))
mask_path = mask_candidates[0]
print('Using mask:', mask_path)


In [ ]:
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.settings.update({'binWidth':25,'normalize':True,'normalizeScale':100,'removeOutliers':True})
extractor.enableAllFeatures()
print('Extractor configured')


In [ ]:
img_sitk = sitk.ReadImage(str(in_img))
mask_sitk = sitk.ReadImage(str(mask_path))
if img_sitk.GetSize() != mask_sitk.GetSize():
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(img_sitk)
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    mask_sitk = resampler.Execute(mask_sitk)
mask_arr = sitk.GetArrayFromImage(mask_sitk)
unique_labels = np.unique(mask_arr)
features_list = []
for label in unique_labels:
    if int(label) == 0:
        continue
    try:
        feats = extractor.execute(img_sitk, mask_sitk, label=int(label))
    except Exception as e:
        print('skip label', label, 'due to', e)
        continue
    feats_filtered = {k:v for k,v in feats.items() if not k.startswith('diagnostics')}
    feats_filtered['Image'] = in_img.stem
    feats_filtered['Label'] = int(label)
    features_list.append(feats_filtered)
if features_list:
    df = pd.DataFrame(features_list)
else:
    df = pd.DataFrame()
out_dir = Path('notebooks/pyradiomic_output')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f'radiomics_{in_img.stem}.csv'
df.to_csv(out_path, index=False)
print('Saved radiomics csv to', out_path)
out_path
